# Random Forest - consolidated best (S6E10 Predicting Airline Satisfaction)

One notebook, one recipe: the strongest **measured** Random Forest configuration from the
whole campaign (45 logged experiments, rounds 1-18, see `experiments.md`). Every choice below
is cited to the experiment that earned it. Nothing speculative is presented as verified.

**Honest status:** Random Forest is not the campaign's best family. This recipe is expected to
score about **0.9577 mean OOF AUC** (Exp 41), roughly **-0.0023 below the preferred candidate**
(Exp 45 LightGBM, 0.960062 mean / 0.960057 pooled; `submission_round18.csv`). If the goal is the
leaderboard, the booster stays the submission candidate. This notebook exists because the best
possible *Random Forest* was requested.

The archived round notebooks under `archive/` carry the full trial-and-error history; this file
is the distilled version.

## Lessons applied (each earned by a logged experiment)

1. **Representation beats tuning.** Fold-local exact value-count features on all 21 raw columns
   (Exp 27) plus split-first, label-blind, cross-fitted smoothed target encoding on all 21
   columns (Exp 31) form the strongest representation measured for every family, RF included
   (Exp 37 measured the RF on exactly this representation).
2. **Leak-free target encoding** (the round-10 invalidation lesson): the split happens before
   any encoding; inner folds are a plain shuffled KFold (label-blind); the smoothing global mean
   is recomputed per inner fold; validation encodings come from tables fit on the outer-train
   rows only. A row's own label never reaches its own encoding.
3. **RF config: 400 trees, min_samples_leaf=5** (Exp 4, the measured RF reference). Measured
   dead ends, not retried here: leaf=10 (Exp 12), max_depth=25 (Exp 13), max_features=0.5
   (Exp 5), ordinal-only on raw features (Exp 15), and the ExtraTrees family on three
   representations (Exp 24, 28, 38).
4. **Seed averaging helps even a bagged model**: averaging seeds 42/43/44 gained +0.000104 over
   the single seed (Exp 41 vs Exp 37, working signal). Small, but real and free.
5. **Same protocol as the whole campaign**: stratified 5-fold OOF on the seed-42 partition, so
   every number here is comparable to the log. Mean and pooled AUC reported separately; paired
   fold deltas with the 2x-spread heuristic (a decision rule, not a significance test).
6. **Forests cannot take NaNs**: the 292 missing Arrival Delay values get a fold-local median
   impute; native categoricals are ordinal-encoded on this representation (Exp 37 construction).
7. **Everything is logged**: OOF arrays + configs + dataset identity are cached to
   `oof_cache/`, and the summary cell prints a ready-made row for `experiments.md`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

RANDOM_STATE = 42
N_SPLITS = 5
SEEDS = [42, 43, 44]          # Exp 41: the measured averaging gain
TE_SMOOTH = 100               # Exp 31/33: m=100 stands; m in {20, 100, 500} all within band
RF_PARAMS = dict(n_estimators=400, min_samples_leaf=5, n_jobs=-1)  # Exp 4 measured reference

# Reference fold AUCs from the log (same seed-42 partition -> paired comparisons are valid)
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # Exp 8 raw LightGBM
E37_FOLDS_R13 = [0.95756, 0.95691, 0.95824, 0.95789, 0.95754]       # single-seed RF, this recipe
E41_FOLDS_R16 = [0.95761, 0.95698, 0.95840, 0.95804, 0.95764]       # 3-seed RF, this exact recipe
E45_FOLDS_R18 = [0.96022, 0.95894, 0.96074, 0.96019, 0.96022]       # preferred booster candidate

competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

In [ ]:
train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
COUNT_COLS = list(features.columns)   # all 21 raw columns (Exp 27)
TE_COLS_ALL = COUNT_COLS              # all 21 target-encoded (Exp 31)
print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features')
print(f'Missing values:\n{features.isna().sum()[features.isna().sum() > 0]}')

## Preprocessing: counts + leak-free cross-fitted TE (the round-12 representation)

This is the audited construction from rounds 10-18, verbatim: fold-local counts, split-first
cross-fitted TE with a per-inner-fold smoothing mean, ordinal-encoded natives, fold-local
median impute. Fitting anything on validation rows is what invalidated the first round-10 run,
so every table below is built from training rows only.

In [ ]:
def add_counts(train_part, other_parts=()):
    # fold-train-only exact-value frequencies; unseen values -> 0 (Exp 27)
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings: a row's own label never enters its encoding - not through the
    # value table and not through the smoothing mean (recomputed per inner fold from itr rows).
    # Inner folds are label-blind (plain shuffled KFold). (Round-10 audit-fixed construction.)
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    # tables fit on reference (training) rows only; validation/test rows never contribute
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def forest_prep(X_tr, y_tr, X_va):
    # counts + leak-free TE, ordinal natives, fold-local median impute (round-13 construction)
    X_tr, (X_va,) = add_counts(X_tr, [X_va])
    te_tr = te_crossfit(X_tr, y_tr, TE_COLS_ALL)
    te_va = te_apply(X_tr, y_tr, X_va, TE_COLS_ALL)
    X_tr = pd.concat([X_tr.reset_index(drop=True), te_tr.reset_index(drop=True)], axis=1)
    X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
    for col in CATEGORICAL:
        cats = pd.Categorical(X_tr[col])
        X_tr[col] = cats.codes
        X_va[col] = pd.Categorical(X_va[col], categories=cats.categories).codes
    med = X_tr.median(numeric_only=True)
    return X_tr.fillna(med), X_va.fillna(med)

In [ ]:
def oof_rf(feat_frame, label, name, seed=RANDOM_STATE):
    oof = np.zeros(len(feat_frame))
    aucs = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        X_tr, X_va = forest_prep(X_tr, y_tr, X_va)
        model = RandomForestClassifier(**RF_PARAMS, random_state=seed)
        model.fit(X_tr, y_tr)
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a)
        print(f'  {name} fold {fold}: AUC {a:.5f}')
    return oof, aucs

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label

## Evaluation: the consolidated recipe, 3 seeds, 5-fold OOF

Expected result from the log: about 0.9577 mean / pooled OOF AUC (Exp 41). The determinism
check compares this run against the round-16 Exp 41 fold constants; PASS means the data and
pipeline reproduce the logged campaign state exactly.

In [ ]:
print(f'Consolidated RF: 400 trees / leaf 5, counts + leak-free TE, seeds {SEEDS}')
oof_seeds = {}
for seed in SEEDS:
    o, a = oof_rf(features, target, f'RF-s{seed}', seed=seed)
    oof_seeds[seed] = o
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')

oof_avg = np.mean([oof_seeds[s] for s in SEEDS], axis=0)
aucs_avg = fold_aucs(oof_avg, target)
mean_oof, pooled_oof = float(np.mean(aucs_avg)), float(roc_auc_score(target, oof_avg))
print(f'\nAveraged fold AUCs: {[round(a, 5) for a in aucs_avg]}')
print(f'Mean OOF AUC: {mean_oof:.6f} +/- {np.std(aucs_avg):.5f} | pooled: {pooled_oof:.6f}')

# Determinism: seed-42 folds should match the round-13 Exp 37 constants, and the
# 3-seed average should match the round-16 Exp 41 constants (same recipe, same data).
diffs_single = [abs(a - b) for a, b in zip(fold_aucs(oof_seeds[42], target), E37_FOLDS_R13)]
diffs_avg = [abs(a - b) for a, b in zip(aucs_avg, E41_FOLDS_R16)]
det_single = 'PASS' if max(diffs_single) < 1e-4 else 'WARN'
det_avg = 'PASS' if max(diffs_avg) < 1e-4 else 'WARN'
print(f'\nDeterminism vs Exp 37 (single-seed RF): max abs diff {max(diffs_single):.6f} - {det_single}')
print(f'Determinism vs Exp 41 (3-seed RF):      max abs diff {max(diffs_avg):.6f} - {det_avg}')
if det_single == 'WARN' or det_avg == 'WARN':
    print('  WARN: numbers diverge from the logged campaign - trust in-run numbers only, and check data identity.')

In [ ]:
print('Paired comparisons (same seed-42 partition as the log):')
m_, s_, lab = paired_verdict(aucs_avg, E37_FOLDS_R13)
print(f'  vs Exp 37 single-seed RF:   mean delta {m_:+.6f} (spread {s_:.5f}) - {lab}')
print('     ^ re-measures the seed-averaging gain (logged at +0.000104 in round 16)')
m_, s_, lab = paired_verdict(aucs_avg, INCUMBENT_FOLDS_R3)
print(f'  vs Exp 8 incumbent booster: mean delta {m_:+.6f} (spread {s_:.5f}) - {lab}')
m_bar, s_bar, lab_bar = paired_verdict(aucs_avg, E45_FOLDS_R18)
print(f'  vs Exp 45 preferred booster: mean delta {m_bar:+.6f} (spread {s_bar:.5f}) - {lab_bar}')
print('\nExpected: the RF stays below the booster. The RF is delivered because it was requested,')
print('not because it wins; submission_round18.csv remains the preferred candidate (see experiments.md).')

## Full-train refit and submission file

Refit the same recipe on all training rows (full-train counts, cross-fitted TE on train,
tables applied to test), average the 3 seeds, and write a local CSV. **Not submitted -
Andrea submits.**

In [ ]:
test = pd.read_csv(data_dir / 'test.csv')
test_features = test.drop(columns=['id'])

X_full, X_test = forest_prep(features, target, test_features)
preds, fitted = [], []
for seed in SEEDS:
    rf = RandomForestClassifier(**RF_PARAMS, random_state=seed)
    rf.fit(X_full, target)
    fitted.append(rf)
    preds.append(rf.predict_proba(X_test)[:, 1])
p = np.mean(preds, axis=0)

sub = pd.DataFrame({'id': test['id'], 'satisfaction': p})
assert len(sub) == len(test) and sub['id'].equals(test['id'])
submission_path = competition_dir / 'submission_rf_consolidated.csv'
sub.to_csv(submission_path, index=False)
import hashlib
md5 = hashlib.md5(submission_path.read_bytes()).hexdigest()
print(f'{submission_path.name} written: {len(sub):,} rows, md5 {md5}')
print('Local file only - NOT submitted; Andrea submits.')

In [ ]:
# What the forest relies on: mean impurity importance across the 3 refit seeds
importances = np.mean([f.feature_importances_ for f in fitted], axis=0)
importance_table = (
    pd.DataFrame({'feature': X_full.columns, 'importance': importances})
    .sort_values('importance', ascending=False)
    .head(15)
)
importance_table

In [ ]:
# OOF cache (established campaign pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(data_dir / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'rf_consolidated_oof.npz',
         **{f'seed_{s}': oof_seeds[s] for s in SEEDS},
         averaged=oof_avg, target=target.to_numpy())
meta = {
    'notebook': 'random_forest_consolidated.ipynb',
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'config': {'model': 'RandomForestClassifier', 'n_estimators': 400, 'min_samples_leaf': 5,
               'seeds': SEEDS, 'representation': 'fold-local counts on 21 raw cols + split-first cross-fitted TE (m=100) on 21 cols, ordinal natives, fold-local median impute'},
    'fold_aucs': {'averaged': aucs_avg},
    'mean_oof': mean_oof, 'pooled_oof': pooled_oof,
    'determinism': {'vs_e37': det_single, 'vs_e41': det_avg},
}
(cache_dir / 'rf_consolidated_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/rf_consolidated_oof.npz + rf_consolidated_oof.json')

## Summary and logging

Fill in the printed numbers and add a row to `experiments.md` (every run gets a row, including
reruns). Suggested entry shape:

| id | date | model / change | config | seed | validation | val AUC | LB AUC | runtime | verdict / notes |
|----|------|----------------|--------|------|------------|---------|--------|---------|-----------------|
| 46 | <date> | consolidated RF notebook (rerun of the Exp 41 recipe) | RF 400/leaf5, counts + leak-free TE, 3-seed average | 42/43/44 | 5-fold OOF | <mean> +/- <std>, pooled <pooled> | | <wall> | determinism <PASS/WARN> vs Exp 37/41 constants; expected ~0.9577, ~-0.0023 below the Exp 45 booster - RF delivered on request, booster stays preferred |

**Where things stand:** the booster campaign ended at Exp 45 (0.960062 mean OOF,
`submission_round18.csv`, written locally on the Mac, not submitted). This RF recipe is the
strongest measured forest and is about 0.0023 behind. Any new RF idea should clear the paired
band against the in-run E41 folds before it earns a place here.